# Repo Characteristics Table
This notebook builds one row per repository and extracts characteristics from multiple local data sources:
- GitHub repository metadata (`data/repositories.json`)
- Commit history snapshots (`data/commits_history/*.json`)
- Language distributions (`data/languages.json`)
- File tree snapshots (`data/file_trees/*.json`)
- README content snapshots (`data/readmes/*.json`)
- Contributors count (`data/contributors/*.json`)
Design principles:
- Batch processing by source, then merge DataFrames
- No per-repo nested joins (linear scans where possible)
- Reusable helper functions
- Progress tracking with `tqdm`

In [1]:
# Core imports
import json
import re
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [2]:
# Paths
ROOT = Path.cwd()
DATA_DIR = ROOT / "../data"

REPOSITORIES_PATH = DATA_DIR / "repositories.json"
LANGUAGES_PATH = DATA_DIR / "languages.json"
COMMITS_DIR = DATA_DIR / "commits_history"
FILE_TREES_DIR = DATA_DIR / "file_trees"
READMES_DIR = DATA_DIR / "readmes"
CONTRIBUTORS_DIR = DATA_DIR / "contributors"

OUTPUT_PATH = DATA_DIR / "repo_characteristics.csv"
print(f"Data directory: {DATA_DIR}")

Data directory: /Users/ningzhi_tang/Documents/VSCodeProjects/empirical-conversational-programming/supplementary_stats/../data


## 1) Repository Metadata Features

In [3]:
# Load repository metadata from GitHub API snapshots
with REPOSITORIES_PATH.open("r", encoding="utf-8") as f:
    repositories_raw = json.load(f)

repos_df = pd.json_normalize(repositories_raw)

# Normalize timestamps and compute repo age in days
for col in ["created_at", "pushed_at", "updated_at"]:
    if col in repos_df.columns:
        repos_df[col] = pd.to_datetime(repos_df[col], utc=True, errors="coerce")

repos_df["repo_age_days"] = (repos_df["pushed_at"] - repos_df["created_at"]).dt.days
repos_df["repo_age_days"] = repos_df["repo_age_days"].clip(lower=0)

# Metadata-derived boolean and count features
license_spdx = (
    repos_df["license.spdx_id"]
    if "license.spdx_id" in repos_df.columns
    else pd.Series([None] * len(repos_df), index=repos_df.index)
)
license_name = (
    repos_df["license.name"]
    if "license.name" in repos_df.columns
    else pd.Series([None] * len(repos_df), index=repos_df.index)
)

repos_df["license_type"] = license_spdx.fillna(license_name)
repos_df["has_license"] = repos_df["license_type"].notna()

repos_df["topics_count"] = repos_df["topics"].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)
repos_df["has_topics"] = repos_df["topics_count"] > 0

repos_df["has_description"] = repos_df["description"].fillna("").str.strip().ne("")

# Keep requested metadata fields
repo_meta_features = repos_df[
    [
        "id",
        "full_name",
        "stargazers_count",
        "forks_count",
        "watchers_count",
        "open_issues_count",
        "size",
        "created_at",
        "pushed_at",
        "repo_age_days",
        "language",
        "has_wiki",
        "has_discussions",
        "has_issues",
        "has_projects",
        "has_license",
        "license_type",
        "fork",
        "archived",
        "has_topics",
        "topics_count",
        "has_description",
    ]
].copy()

# Rename columns to target names
repo_meta_features = repo_meta_features.rename(
    columns={"id": "repo_id", "stargazers_count": "stars", "forks_count": "forks"}
)

repo_meta_features.head()

,repo_id,full_name,stars,forks,watchers_count,open_issues_count,size,created_at,pushed_at,repo_age_days,...,has_discussions,has_issues,has_projects,has_license,license_type,fork,archived,has_topics,topics_count,has_description
0,1084249010,fwornle/coding,2,2,2,1,295537,2025-10-27 12:34:15+00:00,2026-03-04 16:27:29+00:00,128,...,False,True,True,True,MIT,False,False,False,0,True
1,1158681864,The-Raedical-Co/coding-essentials,0,0,0,2,147841,2026-02-15 18:56:36+00:00,2026-02-18 19:53:30+00:00,3,...,False,True,True,True,MIT,False,False,False,0,False
2,17054948,miataru/miataru-ios-client,17,8,17,0,1207436,2014-02-21 11:54:33+00:00,2026-03-04 16:38:11+00:00,4394,...,False,True,True,True,BSD-2-Clause,False,False,False,0,True
3,964416014,Neurotypic-ai/magus-mark,0,1,0,1,27558,2025-04-11 07:17:25+00:00,2026-01-20 19:15:31+00:00,284,...,False,True,True,True,MIT,False,False,True,4,True
4,1039451498,fwornle/agentic-ai-nano,9,3,9,0,112907,2025-08-17 09:01:54+00:00,2026-02-09 10:00:01+00:00,176,...,False,True,True,True,MIT,False,False,True,5,True


## 2) Commit History Features

In [4]:
def extract_commit_dates(commit_items):
    """Extract commit author dates from commit API payload list."""
    dates = []
    if not isinstance(commit_items, list):
        return dates

    for item in commit_items:
        if not isinstance(item, dict):
            continue
        date_str = item.get("commit", {}).get("author", {}).get("date")
        if date_str:
            dates.append(date_str)
    return dates


commit_rows = []
commit_files = sorted(COMMITS_DIR.glob("*.json"))

for path in tqdm(commit_files, desc="Parsing commit history files"):
    repo_id = int(path.stem)

    try:
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)
    except Exception:
        payload = None

    date_values = extract_commit_dates(payload)
    if len(date_values) == 0:
        commit_rows.append(
            {
                "repo_id": repo_id,
                "total_commits": 0,
                "active_days": 0,
                "first_commit_at": pd.NaT,
                "last_commit_at": pd.NaT,
                "commit_span_days": np.nan,
            }
        )
        continue

    dates = pd.to_datetime(pd.Series(date_values), utc=True, errors="coerce").dropna()
    total_commits = int(dates.shape[0])

    if total_commits == 0:
        commit_rows.append(
            {
                "repo_id": repo_id,
                "total_commits": 0,
                "active_days": 0,
                "first_commit_at": pd.NaT,
                "last_commit_at": pd.NaT,
                "commit_span_days": np.nan,
            }
        )
        continue

    first_commit = dates.min()
    last_commit = dates.max()
    span_days = (last_commit - first_commit).days

    # Use at least 1 day to avoid divide-by-zero for same-day commits
    span_days_for_rate = max(span_days, 1)

    commit_rows.append(
        {
            "repo_id": repo_id,
            "total_commits": total_commits,
            "active_days": int(dates.dt.floor("D").nunique()),
            "first_commit_at": first_commit,
            "last_commit_at": last_commit,
            "commit_span_days": float(span_days),
        }
    )

commit_features = pd.DataFrame(commit_rows)
commit_features.head()

Parsing commit history files: 100%|██████████| 1356/1356 [00:09<00:00, 141.35it/s]


,repo_id,total_commits,active_days,first_commit_at,last_commit_at,commit_span_days
0,1000090305,10,1,2025-06-11 07:59:47+00:00,2025-06-11 09:01:07+00:00,0.0
1,1000556517,22,2,2025-06-11 04:08:59+00:00,2025-06-12 01:18:34+00:00,0.0
2,1000596632,6,5,2025-06-12 05:31:10+00:00,2025-06-18 11:49:21+00:00,6.0
3,1000669632,154,19,2025-05-06 01:22:04+00:00,2025-07-01 11:23:26+00:00,56.0
4,1000697395,24,11,2025-05-20 04:20:19+00:00,2026-02-06 05:02:55+00:00,262.0


## 3) Language Distribution Features

In [5]:
def shannon_entropy(values):
    """Compute Shannon entropy from non-negative byte counts."""
    arr = np.asarray(values, dtype=float)
    arr = arr[arr > 0]
    if arr.size == 0:
        return 0.0
    probs = arr / arr.sum()
    return float(-(probs * np.log(probs)).sum())


with LANGUAGES_PATH.open("r", encoding="utf-8") as f:
    languages_raw = json.load(f)

lang_rows = []
for item in tqdm(languages_raw, desc="Parsing language distributions"):
    full_name = item.get("full_name")
    lang_map = item.get("languages") if isinstance(item, dict) else None

    if not isinstance(lang_map, dict) or len(lang_map) == 0:
        lang_rows.append(
            {
                "full_name": full_name,
                "num_languages": 0,
                "primary_language_ratio": np.nan,
                "language_entropy": np.nan,
            }
        )
        continue

    counts = np.array(list(lang_map.values()), dtype=float)
    total = counts.sum()

    if total <= 0:
        lang_rows.append(
            {
                "full_name": full_name,
                "num_languages": 0,
                "primary_language_ratio": np.nan,
                "language_entropy": np.nan,
            }
        )
        continue

    primary_ratio = float(counts.max() / total)

    lang_rows.append(
        {
            "full_name": full_name,
            "num_languages": int((counts > 0).sum()),
            "primary_language_ratio": primary_ratio,
            "language_entropy": shannon_entropy(counts),
        }
    )

language_features = pd.DataFrame(lang_rows)
language_features.head()

Parsing language distributions: 100%|██████████| 1356/1356 [00:00<00:00, 32272.66it/s]


,full_name,num_languages,primary_language_ratio,language_entropy
0,fwornle/coding,8,0.688377,0.877987
1,The-Raedical-Co/coding-essentials,38,0.452057,1.354628
2,miataru/miataru-ios-client,6,0.983175,0.099588
3,Neurotypic-ai/magus-mark,6,0.898707,0.447525
4,fwornle/agentic-ai-nano,6,0.946967,0.245683


## 4) File Tree Structure Features

In [6]:
TEST_PATTERN = re.compile(
    r"(?:^|/)(test|tests|spec|specs)(?:/|$)|(?:^|/).*(?:\.test\.|\.spec\.|_test\.)",
    re.IGNORECASE,
)

CI_MARKERS = {
    ".travis.yml",
    ".gitlab-ci.yml",
    "azure-pipelines.yml",
    "circle.yml",
    ".circleci/config.yml",
}

CONFIG_MARKERS = {
    ".env",
    ".env.example",
    ".env.local",
    "pyproject.toml",
    "setup.py",
    "setup.cfg",
    "requirements.txt",
    "package.json",
    "pom.xml",
    "cargo.toml",
    "makefile",
    "cmakelists.txt",
    "tsconfig.json",
}


def normalize_paths(paths):
    cleaned = []
    for p in paths:
        if isinstance(p, str):
            s = p.strip().strip("/")
            if s:
                cleaned.append(s)
    return cleaned


def derive_dirs_and_files(paths):
    # Build a set of all implied directories from parents in O(total path parts)
    implied_dirs = set()

    for p in paths:
        parts = p.split("/")
        for i in range(1, len(parts)):
            implied_dirs.add("/".join(parts[:i]))

    dirs = implied_dirs
    files = [p for p in paths if p not in dirs]
    return dirs, files


def file_tree_features_from_paths(paths):
    if not isinstance(paths, list):
        return {
            "total_files": 0,
            "total_dirs": 0,
            "max_depth": 0,
            "avg_files_per_dir": np.nan,
            "has_tests": False,
            "has_ci": False,
            "has_dockerfile": False,
            "has_config_files": False,
            "doc_files_count": 0,
        }

    normalized = normalize_paths(paths)
    if len(normalized) == 0:
        return {
            "total_files": 0,
            "total_dirs": 0,
            "max_depth": 0,
            "avg_files_per_dir": np.nan,
            "has_tests": False,
            "has_ci": False,
            "has_dockerfile": False,
            "has_config_files": False,
            "doc_files_count": 0,
        }

    dirs, files = derive_dirs_and_files(normalized)
    file_set = set(files)

    max_depth = max(len(p.split("/")) for p in normalized) if normalized else 0
    total_files = len(files)
    total_dirs = len(dirs)

    avg_files_per_dir = (total_files / total_dirs) if total_dirs > 0 else np.nan

    # Single-pass scan for path-based indicators
    has_tests = False
    has_ci = False
    has_dockerfile = False
    has_config_files = False
    doc_files_count = 0
    doc_files = []  # Collect doc file paths for potential further analysis

    for p in normalized:
        p_lower = p.lower()
        name = p_lower.rsplit("/", 1)[-1]
        is_file = p in file_set

        if is_file:
            if name.startswith("dockerfile"):
                has_dockerfile = True
            if p_lower.endswith(".md"):
                doc_files_count += 1
                doc_files.append(p)

        if not has_tests and TEST_PATTERN.search(p_lower) is not None:
            has_tests = True

        if not has_ci and (p.startswith(".github/workflows/") or p_lower in CI_MARKERS):
            has_ci = True

        if not has_config_files and (
            p_lower.startswith("config/") or name in CONFIG_MARKERS
        ):
            has_config_files = True

        # Early stop when all boolean flags are found (doc count still needs full scan)
        # Keep scanning to finish doc_files_count accurately.

    return {
        "total_files": total_files,
        "total_dirs": total_dirs,
        "max_depth": max_depth,
        "avg_files_per_dir": avg_files_per_dir,
        "has_tests": has_tests,
        "has_ci": has_ci,
        "has_dockerfile": has_dockerfile,
        "has_config_files": has_config_files,
        "doc_files_count": doc_files_count,
        "doc_files": doc_files,
    }


tree_rows = []
tree_files = sorted(FILE_TREES_DIR.glob("*.json"))

for path in tqdm(tree_files, desc="Parsing file tree snapshots"):
    repo_id = int(path.stem)

    try:
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)
    except Exception:
        payload = []

    features = file_tree_features_from_paths(payload)
    features["repo_id"] = repo_id
    tree_rows.append(features)

file_tree_features = pd.DataFrame(tree_rows)
file_tree_features.head()

Parsing file tree snapshots: 100%|██████████| 1356/1356 [00:06<00:00, 218.50it/s]


,total_files,total_dirs,max_depth,avg_files_per_dir,has_tests,has_ci,has_dockerfile,has_config_files,doc_files_count,doc_files,repo_id
0,50,13,3,3.846154,True,False,False,True,8,[.specstory/history/2025-06-04_00-56-如何运行paddl...,1000090305
1,86,19,3,4.526316,False,False,False,False,38,"[.specstory/.what-is-this.md, .specstory/histo...",1000556517
2,136,37,8,3.675676,False,False,False,False,4,[.specstory/history/2025-05-22_06-42-图像分割项目中的标...,1000596632
3,38,23,11,1.652174,False,False,False,True,10,[.specstory/history/2025-06-12_09-50-website-d...,1000669632
4,118,40,12,2.950000,False,False,False,False,5,[spotday/.specstory/history/2025-05-17_23-40Z-...,1000697395


## 5) README Features

In [7]:
# Extract README features (adapted for .md files)
readme_rows = []
readme_files = sorted(READMES_DIR.glob("*.md"))
for path in tqdm(readme_files, desc="Parsing README .md files"):
    repo_id = int(path.stem)
    try:
        with path.open("r", encoding="utf-8") as f:
            text = f.read()
    except Exception:
        text = None
    readme_rows.append(
        {
            "repo_id": repo_id,
            "readme_char_count": len(text) if text is not None else 0,
        }
    )
readme_features = pd.DataFrame(readme_rows)
readme_features.head()

Parsing README .md files: 100%|██████████| 1145/1145 [00:00<00:00, 2372.10it/s]


,repo_id,readme_char_count
0,1000090305,2576
1,1000669632,32350
2,1000783971,5983
3,1000979661,4415
4,1000991563,44157


## 6) Merge All Feature Tables
Also computes the number of contributors per repository from the `contributors` folder.

In [8]:
# Start from repository metadata as the base table
final_df = repo_meta_features.copy()

# Merge source-specific feature tables
final_df = final_df.merge(commit_features, on="repo_id", how="left")
final_df = final_df.merge(language_features, on="full_name", how="left")
final_df = final_df.merge(file_tree_features, on="repo_id", how="left")
final_df = final_df.merge(readme_features, on="repo_id", how="left")

# Compute contributors_count per repo_id
contrib_files = sorted(CONTRIBUTORS_DIR.glob("*.json"))
contrib_count_map = {}
for path in contrib_files:
    repo_id = int(path.stem)
    try:
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)
    except Exception:
        payload = []
    # Count contributors (assume list of dicts or usernames)
    if isinstance(payload, list):
        contrib_count_map[repo_id] = len(payload)
    else:
        contrib_count_map[repo_id] = 0
final_df["contributors_count"] = (
    final_df["repo_id"].map(contrib_count_map).fillna(0).astype(int)
)

# Set has_readme based on presence in readme_features
final_df["has_readme"] = final_df["readme_char_count"] > 0

# Fill selected missing values for count/flag columns first (important for stable derived metrics)
count_cols = [
    "total_commits",
    "active_days",
    "num_languages",
    "total_files",
    "total_dirs",
    "max_depth",
    "doc_files_count",
    "readme_char_count",
    "topics_count",
    "contributors_count",
]
for col in count_cols:
    if col in final_df.columns:
        final_df[col] = final_df[col].fillna(0)

bool_cols = [
    "has_wiki",
    "has_discussions",
    "has_issues",
    "has_projects",
    "has_license",
    "fork",
    "archived",
    "has_topics",
    "has_description",
    "has_readme",
    "has_tests",
    "has_ci",
    "has_dockerfile",
    "has_config_files",
]
for col in bool_cols:
    if col in final_df.columns:
        final_df[col] = final_df[col].fillna(False)

# Fill type-like columns
if "license_type" in final_df.columns:
    final_df["license_type"] = final_df["license_type"].fillna("NO_LICENSE")

# Optional column ordering for readability
ordered_cols = [
    "repo_id",
    "full_name",
    "stars",
    "forks",
    "watchers_count",
    "open_issues_count",
    "size",
    "created_at",
    "pushed_at",
    "repo_age_days",
    "language",
    "has_wiki",
    "has_discussions",
    "has_issues",
    "has_projects",
    "has_license",
    "license_type",
    "fork",
    "archived",
    "has_topics",
    "topics_count",
    "has_description",
    "total_commits",
    "active_days",
    "first_commit_at",
    "last_commit_at",
    "commit_span_days",
    "num_languages",
    "primary_language_ratio",
    "language_entropy",
    "total_files",
    "total_dirs",
    "max_depth",
    "avg_files_per_dir",
    "has_readme",
    "has_tests",
    "has_ci",
    "has_dockerfile",
    "has_config_files",
    "doc_files_count",
    "doc_files",
    "readme_char_count",
    "contributors_count",
]
ordered_cols = [c for c in ordered_cols if c in final_df.columns]
final_df = final_df[
    ordered_cols + [c for c in final_df.columns if c not in ordered_cols]
]

print("Final table shape:", final_df.shape)
final_df.head()

Final table shape: (1356, 43)


,repo_id,full_name,stars,forks,watchers_count,open_issues_count,size,created_at,pushed_at,repo_age_days,...,avg_files_per_dir,has_readme,has_tests,has_ci,has_dockerfile,has_config_files,doc_files_count,doc_files,readme_char_count,contributors_count
0,1084249010,fwornle/coding,2,2,2,1,295537,2025-10-27 12:34:15+00:00,2026-03-04 16:27:29+00:00,128,...,16.502283,True,True,True,True,True,1989,"[.claude-resume/llm-docs-update-complete.md, ....",17693.0,1
1,1158681864,The-Raedical-Co/coding-essentials,0,0,0,2,147841,2026-02-15 18:56:36+00:00,2026-02-18 19:53:30+00:00,3,...,8.215412,True,True,True,True,True,2010,"[.agent/experts/_template/build-agent.md, .age...",17608.0,1
2,17054948,miataru/miataru-ios-client,17,8,17,0,1207436,2014-02-21 11:54:33+00:00,2026-03-04 16:38:11+00:00,4394,...,5.170297,True,True,False,True,False,458,"[3rd party licenses.md, README.md, documentati...",10655.0,1
3,964416014,Neurotypic-ai/magus-mark,0,1,0,1,27558,2025-04-11 07:17:25+00:00,2026-01-20 19:15:31+00:00,284,...,7.068376,True,True,False,False,True,388,"[.specstory/.what-is-this.md, .specstory/histo...",4532.0,2
4,1039451498,fwornle/agentic-ai-nano,9,3,9,0,112907,2025-08-17 09:01:54+00:00,2026-02-09 10:00:01+00:00,176,...,18.923077,True,True,True,True,True,753,"[.claude/agents/course-material-enhancer.md, ....",15332.0,1


In [9]:
# Save results
final_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")

Saved: /Users/ningzhi_tang/Documents/VSCodeProjects/empirical-conversational-programming/supplementary_stats/../data/repo_characteristics.csv


In [10]:
missing_ratio = final_df.isna().mean()
print(missing_ratio[missing_ratio > 0].sort_values(ascending=False))

language                  0.026549
primary_language_ratio    0.025074
language_entropy          0.025074
dtype: float64
